# 09 — Seed-bagging CatBoost (push vers la 1re place)

On est 2e (LB 0.3569), à 0.0007 du 1er (0.3576). Modèle = CatBoost, features du 08,
smoothing=30, **sans calibration**. On moyenne plusieurs graines pour gratter du signal stable.

Boussole : **LB ≈ CV last − 0.004**. Si le bag pousse le last fold > 0.362, on vise la 1re place.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from src import config as C
from src.validation import time_folds, evaluate_ap
from src.utils import op03_mask, seed_everything, make_submission
from src.features.temporal import balance_features, recency_features
from src.features.behavioral import behavioral_features
from src.encoding import oof_target_encode_train, fit_target_map, apply_target_map, recent_target_rate
seed_everything(42)
DATA = ROOT / "data"
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
op03 = op03_mask(train).to_numpy()
y_all = train[C.TARGET].to_numpy()
folds_full = list(time_folds(train[C.PERIOD]))

In [ ]:
EPS = 1e-6
WINDOWS = (5, 10, 20)
SMOOTHING = 30
SEEDS = [42, 1, 7, 13, 99]
ITERS = 800

def row_features(df):
    f = pd.DataFrame(index=df.index)
    f["amount_log1p"] = np.log1p(np.maximum(df[C.AMOUNT], 0))
    f["amount_vs_origin_before"] = df[C.AMOUNT] / (np.abs(df[C.ORIGIN_BAL_BEFORE]) + EPS)
    f["amount_vs_dest_before"] = df[C.AMOUNT] / (np.abs(df[C.DEST_BAL_BEFORE]) + EPS)
    f["origin_balance_before"] = df[C.ORIGIN_BAL_BEFORE]
    f["dest_balance_before"] = df[C.DEST_BAL_BEFORE]
    return pd.concat([f, balance_features(df)], axis=1)

def add_freq(X, src_df, ref_df):
    X = X.copy()
    for col in [C.ORIGIN_ACCT, C.DEST_ACCT]:
        freq = ref_df[col].value_counts(normalize=True)
        X[f"freq_{col}"] = src_df[col].map(freq).fillna(0).values
    return X

def base_build(df, ref):
    X = row_features(df).reset_index(drop=True)
    X = add_freq(X, df.reset_index(drop=True), ref)
    beh = behavioral_features(df, ref).reset_index(drop=True)
    rec = recency_features(df, ref).reset_index(drop=True)
    rt = recent_target_rate(df, ref, C.ORIGIN_ACCT, C.PERIOD, C.TARGET, WINDOWS).reset_index(drop=True)
    return pd.concat([X, beh, rec, rt], axis=1)

def feats_train(df, ref):
    X = base_build(df, ref)
    X["te_origin"] = oof_target_encode_train(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SMOOTHING)
    return X

def feats_apply(df, ref):
    X = base_build(df, ref)
    mp, gm = fit_target_map(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SMOOTHING)
    X["te_origin"] = apply_target_map(df, C.ORIGIN_ACCT, mp, gm)
    return X

def make_cat(seed):
    from catboost import CatBoostClassifier
    return CatBoostClassifier(loss_function="Logloss", eval_metric="PRAUC", depth=6,
                              learning_rate=0.05, iterations=ITERS, random_seed=seed, verbose=False)

## CV : 1 graine vs bag de 5 graines (features construites une fois par fold)

In [ ]:
oof_single = np.zeros(len(train)); oof_bag = np.zeros(len(train))
ap_single, ap_bag = [], []
for tr_idx, va_idx in folds_full:
    tr_op = tr_idx[op03[tr_idx]]; va_op = va_idx[op03[va_idx]]
    ref = train.iloc[tr_op]
    Xtr = feats_train(train.iloc[tr_op], ref)
    Xva = feats_apply(train.iloc[va_op], ref)
    yt = y_all[tr_op]
    preds = []
    for s in SEEDS:
        p = make_cat(s).fit(Xtr, yt).predict_proba(Xva)[:, 1]
        preds.append(p)
        if s == SEEDS[0]:
            oof_single[va_op] = p
    oof_bag[va_op] = np.mean(preds, axis=0)
    ap_single.append(evaluate_ap(y_all[va_op], oof_single[va_op]))
    ap_bag.append(evaluate_ap(y_all[va_op], oof_bag[va_op]))
    print(f"fold: single {ap_single[-1]:.4f} | bag {ap_bag[-1]:.4f}")

def show(name, pf):
    print(f"{name:8s} recent(2) {np.mean(pf[-2:]):.4f} | last {pf[-1]:.4f} | LB~ {pf[-1]-0.004:.4f}")
print("\n--- Synthèse (LB~ = last - 0.004, calibrée sur le 08) ---")
show("single", ap_single); show("bag", ap_bag)
print("\nGain bag sur last fold :", round(ap_bag[-1] - ap_single[-1], 4))

## Soumission du bag (sans calibration) — DERNIÈRE DU JOUR
Uploader seulement si `LB~ (bag)` >= 0.3569 (au moins égaler notre 2e place).

In [ ]:
ref_full = train.iloc[np.where(op03)[0]]
Xf = feats_train(ref_full, ref_full); yf = y_all[op03]
te_op = op03_mask(test).to_numpy()
test_op = test.iloc[np.where(te_op)[0]]
Xte = feats_apply(test_op, ref_full)

preds = [make_cat(s).fit(Xf, yf).predict_proba(Xte)[:, 1] for s in SEEDS]
proba = np.mean(preds, axis=0)
full = np.zeros(len(test)); full[te_op] = proba
path = make_submission(test[C.ID], full, "09_bag5")
sub = pd.read_csv(path)
assert list(sub.columns) == ["id", "target"] and len(sub) == len(test)
assert set(sub["id"]) == set(sample["id"]) and sub["target"].between(0, 1).all()
print("soumission écrite :", path, "| proba>0 :", int((sub['target'] > 0).sum()))